In [ ]:
import pandas as pd

df = pd.read_csv(
    "historique-consommation-secteur-activite-mun-mois.csv",
    sep=None,
    engine="python"
)

df

In [ ]:
import pandas as pd

df_geo = pd.read_excel(
    "Quebec_1152_locations_geocoded.xlsx"
)

df_geo

In [ ]:
df = df.merge(
    df_geo[
        ["NOM_MUNICIPALITE", "LATITUDE", "LONGITUDE"]
    ],
    on="NOM_MUNICIPALITE",
    how="left"
)

df

In [ ]:
import pandas as pd
import numpy as np

# --- 1. Asegurar tipos ---
df["ANNEE_MOIS"] = pd.to_datetime(df["ANNEE_MOIS"])

# Detectar automáticamente el nombre de la columna de consumo
if "Total (kWh)" in df.columns:
    kwh_col = "Total (kWh)"
elif "total_kwh" in df.columns:
    kwh_col = "total_kwh"
else:
    raise ValueError("No encuentro la columna de consumo eléctrico.")

# --- 2. Sumar energía total por municipio y fecha (sumando sectores) ---
df_muni = (
    df.groupby(
        ["NOM_MUNICIPALITE", "LATITUDE", "LONGITUDE", "ANNEE_MOIS"],
        as_index=False
    )
    .agg(total_kwh=(kwh_col, lambda s: s.sum(min_count=1)))
)

# --- 3. Fechas globales mínima y máxima ---
min_date = df_muni["ANNEE_MOIS"].min()
max_date = df_muni["ANNEE_MOIS"].max()

print("Fecha mínima:", min_date)
print("Fecha máxima:", max_date)

# --- 4. Extraer municipio en fecha inicial y final ---
df_start = (
    df_muni[df_muni["ANNEE_MOIS"] == min_date]
    [["NOM_MUNICIPALITE", "LATITUDE", "LONGITUDE", "total_kwh"]]
    .rename(columns={"total_kwh": "kwh_start"})
)

df_end = (
    df_muni[df_muni["ANNEE_MOIS"] == max_date]
    [["NOM_MUNICIPALITE", "LATITUDE", "LONGITUDE", "total_kwh"]]
    .rename(columns={"total_kwh": "kwh_end"})
)

# --- 5. Unir inicio y fin ---
df_change = df_start.merge(
    df_end[["NOM_MUNICIPALITE", "kwh_end"]],
    on="NOM_MUNICIPALITE",
    how="inner"
)

# --- 6. Variación absoluta y porcentual ---
df_change["change_abs"] = df_change["kwh_end"] - df_change["kwh_start"]

df_change["change_pct"] = np.where(
    (df_change["kwh_start"].notna()) & (df_change["kwh_start"] != 0),
    (df_change["kwh_end"] - df_change["kwh_start"]) / df_change["kwh_start"] * 100,
    np.nan
)

# --- 7. Etiqueta interpretativa ---
df_change["status"] = np.where(
    df_change["change_pct"] > 0, "Creciendo",
    np.where(df_change["change_pct"] < 0, "Apagándose", "Sin cambio")
)

# --- 8. Limpiar observaciones sin coordenadas o sin cambio calculable ---
df_change = df_change.dropna(subset=["LATITUDE", "LONGITUDE", "change_pct"])

df_change.head()

In [ ]:
import plotly.express as px

fig = px.scatter_map(
    df_change,
    lat="LATITUDE",
    lon="LONGITUDE",
    color="change_pct",
    size=df_change["change_pct"].abs(),
    hover_name="NOM_MUNICIPALITE",
    hover_data={
        "kwh_start": ":,.0f",
        "kwh_end": ":,.0f",
        "change_pct": ":.2f",
        "LATITUDE": False,
        "LONGITUDE": False
    },
    zoom=4,
    center={"lat": 52.0, "lon": -71.5},
    map_style="open-street-map",
    title=f"Québec: variación % del consumo total de energía entre {min_date.date()} y {max_date.date()}",
    color_continuous_midpoint=0
)

fig.update_layout(
    width=850,
    height=1000,
    margin={"r": 0, "t": 60, "l": 0, "b": 0}
)

fig.show()

In [ ]:
# fig = px.scatter_map(
#     df_change,
#     lat="LATITUDE",
#     lon="LONGITUDE",
#     color="change_pct",
#     size=df_change["change_pct"].abs(),
#     hover_name="NOM_MUNICIPALITE",
#     hover_data={
#         "kwh_start": ":,.0f",
#         "kwh_end": ":,.0f",
#         "change_pct": ":.2f",
#         "LATITUDE": False,
#         "LONGITUDE": False
#     },
#     zoom=4,
#     center={"lat": 52.0, "lon": -71.5},
#     map_style="carto-positron",
#     title=f"Québec: variación % del consumo total de energía entre {min_date.date()} y {max_date.date()}",
#     color_continuous_midpoint=0
# )

# fig.update_layout(
#     margin={"r": 0, "t": 60, "l": 0, "b": 0}
# )

In [ ]:
# fig.write_html(
#     "quebec_energy_change_map.html",
#     include_plotlyjs="cdn",
#     full_html=True
# )